# LeetCode 3477 — Fruits Into Baskets II bằng Python

Link đề: https://leetcode.com/problems/fruits-into-baskets-ii/ · Độ khó: **Easy** · Chủ đề: Array, Simulation, Segment Tree

Skeleton LeetCode:

```python
class Solution:
    def numOfUnplacedFruits(self, fruits: List[int], baskets: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: duyệt các loại quả **từ trái sang phải**, mỗi loại bỏ vào **giỏ trống nằm ngoài cùng bên trái** mà **sức chứa >= số lượng quả**; mỗi giỏ chỉ chứa **một** loại; đếm số loại quả **không** có giỏ.
2. Mô phỏng bằng hai vòng lặp lồng nhau với mảng đánh dấu `used`, và hiểu vì sao "leftmost" (trái nhất) quan trọng — không phải "vừa khít nhất".
3. Biết hai cách làm gọn hơn: xoá giỏ đã dùng khỏi danh sách, và **segment tree** (cây đoạn) lưu `max` để tìm giỏ trái nhất đủ lớn trong `O(log n)`.
4. Viết 4 cách, đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: dùng lại một giỏ nhiều lần, chọn giỏ nhỏ nhất vừa đủ (best-fit), sắp xếp cả hai mảng rồi ghép, và chỉ so sánh `fruits[i]` với `baskets[i]` cùng vị trí.

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên **cùng độ dài** `n`: `fruits[i]` là **số lượng** quả của loại thứ `i`, `baskets[j]` là **sức chứa** của giỏ thứ `j`.

Xếp quả theo quy tắc: xét các loại quả **từ trái sang phải**. Mỗi loại quả được bỏ vào **giỏ trống (chưa dùng) nằm bên trái nhất** mà có sức chứa **lớn hơn hoặc bằng** số lượng quả loại đó. Mỗi giỏ chỉ chứa **đúng một** loại quả. Nếu không có giỏ nào phù hợp thì loại quả đó **không được xếp** (unplaced).

Trả về **số loại quả không được xếp**.

| Ví dụ | fruits, baskets | Output | Giải thích |
|---|---|---|---|
| 1 | `[4,2,5]`, `[3,5,4]` | 1 | `4` vào giỏ `5`; `2` vào giỏ `3`; `5` không còn giỏ nào >= 5 (còn giỏ `4`) -> 1 loại không xếp |
| 2 | `[3,6,1]`, `[6,4,7]` | 0 | `3` vào giỏ `6`; `6` bỏ qua giỏ `4`, vào giỏ `7`; `1` vào giỏ `4` |

Constraints: `n == fruits.length == baskets.length`; `1 <= n <= 100`; `1 <= fruits[i], baskets[i] <= 1000`.

In [1]:
fruits = [4, 2, 5]
baskets = [3, 5, 4]
used = [False] * len(baskets)
unplaced = 0
for f in fruits:
    placed = False
    for j in range(len(baskets)):
        if not used[j] and baskets[j] >= f:
            used[j] = True
            placed = True
            print(f"quả {f} -> giỏ {j} (sức chứa {baskets[j]})")
            break
    if not placed:
        unplaced += 1
        print(f"quả {f} -> KHÔNG có giỏ")
print("số loại không xếp =", unplaced)

quả 4 -> giỏ 1 (sức chứa 5)
quả 2 -> giỏ 0 (sức chứa 3)
quả 5 -> KHÔNG có giỏ
số loại không xếp = 1


## 3. Kiến thức nền

### 3.1. Hai mảng song song và chỉ số

`fruits` và `baskets` là **hai mảng độc lập** cùng độ dài; chỉ số `i` của `fruits` **không** gắn với chỉ số `i` của `baskets`. Quả `i` có thể vào giỏ `j` bất kỳ `j != i`. Chỉ số của giỏ chỉ quan trọng ở chỗ nó xác định **thứ tự ưu tiên** (giỏ chỉ số nhỏ hơn = "trái hơn" = ưu tiên hơn).

In [2]:
fruits = [4, 2, 5]
baskets = [3, 5, 4]
for i, f in enumerate(fruits):
    print(f"quả {i} có {f} ; giỏ {i} chứa {baskets[i]}")

quả 0 có 4 ; giỏ 0 chứa 3
quả 1 có 2 ; giỏ 1 chứa 5
quả 2 có 5 ; giỏ 2 chứa 4


### 3.2. Mảng đánh dấu `used` và `break`

`used = [False] * n` ghi lại giỏ nào đã bị dùng. Vòng lặp trong tìm giỏ phù hợp đầu tiên; khi tìm được ta đặt `used[j] = True` và **`break`** để không dùng thêm giỏ nào nữa cho cùng loại quả. Có thể dùng `for ... else` hoặc một biến cờ `placed` để biết có tìm thấy không.

In [3]:
for j in range(5):
    if j == 2:
        print("tìm thấy j =", j)
        break
else:
    print("không tìm thấy")          # nhánh else chỉ chạy khi KHÔNG break
for j in range(5):
    if j == 9:
        break
else:
    print("else chạy vì vòng lặp không bị break")

tìm thấy j = 2
else chạy vì vòng lặp không bị break


### 3.3. `next(generator, default)` để tìm phần tử đầu tiên

`next((j for j in range(n) if điều_kiện(j)), -1)` trả về `j` **đầu tiên** thoả điều kiện, hoặc `-1` nếu không có. Gọn hơn hai vòng lặp có cờ.

In [4]:
baskets = [3, 5, 4]
print(next((j for j in range(3) if baskets[j] >= 4), -1))
print(next((j for j in range(3) if baskets[j] >= 9), -1))

1
-1


### 3.4. `list.pop(i)` — xoá phần tử tại vị trí `i`

`lst.pop(i)` xoá và trả về phần tử ở chỉ số `i`, các phần tử sau dịch sang trái. Ta có thể dùng nó để "lấy giỏ ra khỏi danh sách" sau khi dùng, thay cho mảng `used`. Chú ý `pop` tốn `O(n)`.

In [5]:
lst = [3, 5, 4]
j = 1
print(lst.pop(j), lst)

5 [3, 4]


### 3.5. Segment tree lưu `max` (cây đoạn)

Một **segment tree** là cây nhị phân lưu thông tin của từng **đoạn** mảng. Ở đây mỗi nút lưu **giá trị lớn nhất** của đoạn tương ứng. Nhờ vậy ta biết ngay "đoạn này có giỏ nào >= `f` không": chỉ cần `max_của_nút >= f`. Để tìm giỏ **trái nhất** đủ lớn: từ gốc, nếu con **trái** có `max >= f` thì đi trái, ngược lại đi phải. Mất `O(log n)`. Cài bằng mảng `tree` kích thước `2 * size` (nút `i` có con `2i` và `2i + 1`; lá nằm ở `size..2*size-1`).

In [6]:
baskets = [3, 5, 4]
size = 1
while size < len(baskets):
    size *= 2                      # size = 4
tree = [0] * (2 * size)
for i, b in enumerate(baskets):
    tree[size + i] = b             # lá
for i in range(size - 1, 0, -1):
    tree[i] = max(tree[2 * i], tree[2 * i + 1])
print("size =", size, "tree =", tree)
print("giỏ nào lớn nhất trong cả mảng?", tree[1])

size = 4 tree = [0, 5, 5, 4, 3, 5, 4, 0]
giỏ nào lớn nhất trong cả mảng? 5


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Làm tay với `fruits = [4, 2, 5]`, `baskets = [3, 5, 4]`:

```
giỏ:      [3, 5, 4]   (chỉ số 0, 1, 2)
quả 4: giỏ 0 (=3) nhỏ hơn 4 -> bỏ qua; giỏ 1 (=5) >= 4 -> dùng giỏ 1.   giỏ trống: [3, _, 4]
quả 2: giỏ 0 (=3) >= 2 -> dùng giỏ 0.                                 giỏ trống: [_, _, 4]
quả 5: giỏ 2 (=4) < 5 -> không còn giỏ.                               => unplaced = 1
```

Mỗi quả chỉ quan tâm: "giỏ **trống** đầu tiên từ trái có sức chứa >= số lượng". Đó là một phép **tìm kiếm tuyến tính** nhớ trạng thái đã-dùng.

## 5. Cách 1 — Brute Force / baseline

Với mỗi quả `f`, quét `j` từ `0` tới `n - 1`; gặp giỏ chưa dùng có `baskets[j] >= f` thì đánh dấu dùng và dừng. Nếu quét hết mà không gặp thì tăng bộ đếm.

Nút thắt: mỗi quả có thể quét cả `n` giỏ -> `O(n^2)`. Với `n <= 100` là 10 nghìn phép so sánh, quá nhanh; nhưng nếu `n` lớn (bài **Fruits Into Baskets III** có `n` tới `10^5`) thì không đủ.

In [7]:
class SolutionBrute:
    def numOfUnplacedFruits(self, fruits, baskets):
        n = len(baskets)
        used = [False] * n
        unplaced = 0
        for f in fruits:
            placed = False
            for j in range(n):
                if not used[j] and baskets[j] >= f:
                    used[j] = True
                    placed = True
                    break
            if not placed:
                unplaced += 1
        return unplaced

print(SolutionBrute().numOfUnplacedFruits([4, 2, 5], [3, 5, 4]))
print(SolutionBrute().numOfUnplacedFruits([3, 6, 1], [6, 4, 7]))

1
0


**Độ phức tạp:** Time `O(n^2)`, Space `O(n)` cho `used`.

## 6. Ý tưởng thuật toán / bước đột phá

**Điều cần giữ chắc:** quy tắc là **trái nhất đủ lớn**, không phải "nhỏ nhất đủ lớn". Ta không được sắp xếp lại giỏ.

**Bước đột phá (tăng tốc):** thay vì quét tuyến tính từng giỏ, dùng cấu trúc cho phép hỏi "giỏ trái nhất có sức chứa >= `f`" nhanh hơn. Đó là segment tree lưu `max`:

- Nếu `tree[1]` (max toàn mảng) `< f` -> không có giỏ nào, quả này không xếp được.
- Ngược lại đi từ gốc xuống: nếu `tree[2i] >= f` thì đi **trái**, còn không đi **phải** (khi đó chắc chắn con phải có `max >= f`).
- Khi đã dùng giỏ, đặt lá của nó về `0` (không quả nào có số lượng `0` vì `fruits[i] >= 1`) và cập nhật lên gốc.

Invariant: lá bằng `0` nghĩa là giỏ đã dùng (hoặc ô đệm); `tree[i]` luôn bằng `max` các lá trong đoạn của nút `i`.

In [8]:
# thử tay: tìm giỏ trái nhất có sức chứa >= 4 trên cây đã dựng
baskets = [3, 5, 4]
size = 4
tree = [0] * (2 * size)
for i, b in enumerate(baskets):
    tree[size + i] = b
for i in range(size - 1, 0, -1):
    tree[i] = max(tree[2 * i], tree[2 * i + 1])
f = 4
node = 1
while node < size:
    node = 2 * node if tree[2 * node] >= f else 2 * node + 1
    print("đi tới nút", node, "max =", tree[node])
print("giỏ trái nhất >= 4 có chỉ số", node - size)

đi tới nút 2 max = 5
đi tới nút 5 max = 5
giỏ trái nhất >= 4 có chỉ số 1


## 7. Các cách tiếp theo

Ta có 4 cách: (1) hai vòng lặp + `used`; (2) xoá giỏ đã dùng khỏi danh sách bằng `pop`; (3) `next` + cờ đánh dấu bằng `0`; (4) segment tree `max`.

### Cách 2 — Sao chép danh sách giỏ và `pop` giỏ đã dùng

Tạo bản sao `left = list(baskets)`. Với mỗi quả, tìm vị trí đầu tiên `j` có `left[j] >= f` (bằng `next`), nếu có thì `left.pop(j)` (giỏ đó biến mất, các giỏ sau dịch trái **nhưng thứ tự tương đối giữ nguyên**, nên "trái nhất" vẫn đúng).

In [9]:
class SolutionPop:
    def numOfUnplacedFruits(self, fruits, baskets):
        left = list(baskets)
        unplaced = 0
        for f in fruits:
            j = next((i for i, cap in enumerate(left) if cap >= f), -1)
            if j == -1:
                unplaced += 1
            else:
                left.pop(j)
        return unplaced

print(SolutionPop().numOfUnplacedFruits([4, 2, 5], [3, 5, 4]))
print(SolutionPop().numOfUnplacedFruits([3, 6, 1], [6, 4, 7]))

1
0


Độ phức tạp Cách 2: Time `O(n^2)` (tìm `O(n)` + `pop` `O(n)` cho mỗi quả), Space `O(n)`.

### Cách 3 — Đánh dấu giỏ đã dùng bằng `0` (không cần mảng `used`)

Vì `fruits[i] >= 1`, một giỏ có sức chứa `0` không bao giờ thoả `baskets[j] >= f`. Ta **ghi đè** `0` lên giỏ đã dùng (trên bản sao) và chỉ còn một điều kiện `cap >= f`.

In [10]:
class SolutionZeroMark:
    def numOfUnplacedFruits(self, fruits, baskets):
        cap = list(baskets)
        unplaced = 0
        for f in fruits:
            j = next((i for i in range(len(cap)) if cap[i] >= f), -1)
            if j == -1:
                unplaced += 1
            else:
                cap[j] = 0        # giỏ đã dùng: sức chứa 0, không quả nào (>= 1) lọt vào nữa
        return unplaced

print(SolutionZeroMark().numOfUnplacedFruits([4, 2, 5], [3, 5, 4]))
print(SolutionZeroMark().numOfUnplacedFruits([3, 6, 1], [6, 4, 7]))

1
0


Độ phức tạp Cách 3: Time `O(n^2)` trong trường hợp xấu, Space `O(n)` (bản sao).

### Cách 4 — Segment tree lưu `max` (tìm trái nhất trong `O(log n)`)

Dựng cây như mục 3.5; với mỗi quả: nếu `tree[1] < f` thì unplaced; còn không thì đi từ gốc theo quy tắc "con trái đủ lớn thì đi trái", tới lá `pos`, rồi `tree[pos] = 0` và cập nhật các tổ tiên bằng `max` của hai con.

In [11]:
class SolutionSegTree:
    def numOfUnplacedFruits(self, fruits, baskets):
        n = len(baskets)
        size = 1
        while size < n:
            size *= 2
        tree = [0] * (2 * size)
        for i, b in enumerate(baskets):
            tree[size + i] = b
        for i in range(size - 1, 0, -1):
            tree[i] = max(tree[2 * i], tree[2 * i + 1])
        unplaced = 0
        for f in fruits:
            if tree[1] < f:
                unplaced += 1
                continue
            node = 1
            while node < size:
                node = 2 * node if tree[2 * node] >= f else 2 * node + 1
            tree[node] = 0                      # dùng giỏ này
            node //= 2
            while node >= 1:
                tree[node] = max(tree[2 * node], tree[2 * node + 1])
                node //= 2
        return unplaced

print(SolutionSegTree().numOfUnplacedFruits([4, 2, 5], [3, 5, 4]))
print(SolutionSegTree().numOfUnplacedFruits([3, 6, 1], [6, 4, 7]))

1
0


Độ phức tạp Cách 4: Time `O(n log n)` (dựng `O(n)`, mỗi quả `O(log n)`), Space `O(n)`. Với `n <= 100` không cần thiết, nhưng là kỹ thuật chuẩn cho phiên bản `n` lớn.

## 8. TRACE từng bước

Trace Cách 1 với `fruits = [3, 6, 1]`, `baskets = [6, 4, 7]`:

| quả `f` | quét giỏ `j` | kết quả | `used` sau bước | `unplaced` |
|---|---|---|---|---|
| 3 | j=0: `6 >= 3` | dùng giỏ 0 | `[T,F,F]` | 0 |
| 6 | j=0 đã dùng; j=1: `4 < 6`; j=2: `7 >= 6` | dùng giỏ 2 | `[T,F,T]` | 0 |
| 1 | j=0 đã dùng; j=1: `4 >= 1` | dùng giỏ 1 | `[T,T,T]` | 0 |

Kết quả `0`: mọi loại quả đều có giỏ.

In [12]:
def trace_baskets(fruits, baskets):
    n = len(baskets)
    used = [False] * n
    unplaced = 0
    for f in fruits:
        print(f"--- quả {f} ---")
        placed = False
        for j in range(n):
            if used[j]:
                print(f"  j={j}: đã dùng, bỏ qua")
                continue
            ok = baskets[j] >= f
            print(f"  j={j}: sức chứa {baskets[j]} >= {f}? {ok}")
            if ok:
                used[j] = True
                placed = True
                print(f"  => dùng giỏ {j}; used={used}")
                break
        if not placed:
            unplaced += 1
            print(f"  => không có giỏ; unplaced={unplaced}")
    return unplaced

print("kết quả:", trace_baskets([3, 6, 1], [6, 4, 7]))
print("kết quả:", trace_baskets([4, 2, 5], [3, 5, 4]))

--- quả 3 ---
  j=0: sức chứa 6 >= 3? True
  => dùng giỏ 0; used=[True, False, False]
--- quả 6 ---
  j=0: đã dùng, bỏ qua
  j=1: sức chứa 4 >= 6? False
  j=2: sức chứa 7 >= 6? True
  => dùng giỏ 2; used=[True, False, True]
--- quả 1 ---
  j=0: đã dùng, bỏ qua
  j=1: sức chứa 4 >= 1? True
  => dùng giỏ 1; used=[True, True, True]
kết quả: 0
--- quả 4 ---
  j=0: sức chứa 3 >= 4? False
  j=1: sức chứa 5 >= 4? True
  => dùng giỏ 1; used=[False, True, False]
--- quả 2 ---
  j=0: sức chứa 3 >= 2? True
  => dùng giỏ 0; used=[True, True, False]
--- quả 5 ---
  j=0: đã dùng, bỏ qua
  j=1: đã dùng, bỏ qua
  j=2: sức chứa 4 >= 5? False
  => không có giỏ; unplaced=1
kết quả: 1


## 9. Code nộp LeetCode

In [13]:
from typing import List

class Solution:
    def numOfUnplacedFruits(self, fruits: List[int], baskets: List[int]) -> int:
        n = len(baskets)
        used = [False] * n
        unplaced = 0
        for f in fruits:
            placed = False
            for j in range(n):
                if not used[j] and baskets[j] >= f:
                    used[j] = True
                    placed = True
                    break
            if not placed:
                unplaced += 1
        return unplaced

print(Solution().numOfUnplacedFruits([4, 2, 5], [3, 5, 4]), Solution().numOfUnplacedFruits([3, 6, 1], [6, 4, 7]))

1 0


## 10. Vì sao thuật toán đúng

- **Mô phỏng đúng quy tắc:** với mỗi loại quả (đúng thứ tự trái -> phải), ta chọn giỏ **chưa dùng**, **đủ chứa** (`>=`) và có **chỉ số nhỏ nhất** — chính là "bên trái nhất" của đề. Vòng `for j` tăng dần kèm `break` ở giỏ đầu tiên thoả mãn bảo đảm điều đó.
- **Mỗi giỏ một loại:** sau khi chọn, `used[j] = True` nên giỏ không được chọn lại. Nếu không đánh dấu (Lỗi 1), quả sau sẽ lấy lại cùng giỏ.
- **Đếm đúng:** biến `placed` chỉ đúng khi tìm thấy giỏ; nếu quét hết mà không có thì loại quả đó không xếp được và ta cộng `1`. Mỗi loại quả được xét đúng một lần nên tổng không đếm trùng.
- **Cách 2, 3:** xoá hoặc ghi `0` vào giỏ đã dùng tương đương `used[j] = True` (vì `f >= 1` nên `0 >= f` luôn sai; còn `pop` giữ nguyên thứ tự tương đối).
- **Cách 4:** tại mỗi nút, `tree[node]` là max của đoạn; nếu con trái có `max >= f` thì giỏ trái nhất thoả mãn chắc chắn nằm ở con trái (vì mọi chỉ số ở con trái nhỏ hơn con phải); nếu không thì chắc chắn nằm ở con phải (vì nút hiện tại có `max >= f`).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(fruits) = len(baskets)`.

| Cách | Time | Space |
|---|---|---|
| 1. Hai vòng lặp + `used` | `O(n^2)` | `O(n)` |
| 2. Bản sao + `pop` | `O(n^2)` | `O(n)` |
| 3. Đánh dấu `0` + `next` | `O(n^2)` | `O(n)` |
| 4. Segment tree `max` | `O(n log n)` | `O(n)` |

Với `n <= 100` cả bốn đều chạy dưới một mili giây. Segment tree chỉ thật sự cần khi `n` tới `10^5` (bài III).

## 12. Bộ test

Test 1: ví dụ đề và các edge case (`n = 1` vừa đủ / không đủ, tất cả vào được, không quả nào vào được, giỏ chuyển hướng trái nhất, giá trị biên `1000`).

In [14]:
impls = [SolutionBrute(), SolutionPop(), SolutionZeroMark(), SolutionSegTree(), Solution()]
tests = [
    ([4, 2, 5], [3, 5, 4], 1),
    ([3, 6, 1], [6, 4, 7], 0),
    ([3], [3], 0),                             # n = 1, vừa khít (>=)
    ([4], [3], 1),                             # n = 1, thiếu
    ([1000], [1000], 0),                       # biên trên
    ([1], [1000], 0),                          # quả nhỏ vào giỏ rất lớn
    ([1000], [1], 1),                          # quả lớn không vào giỏ nhỏ
    ([2, 2], [3, 1], 1),                       # giỏ 3 dùng một lần, giỏ 1 không đủ (bắt lỗi dùng lại giỏ)
    ([3, 5], [5, 4], 1),                       # trái nhất: 3->giỏ 5 (không phải giỏ 4); 5 không còn giỏ
    ([5, 5, 5], [1, 2, 3], 3),                 # không quả nào vào được
    ([1, 1, 1], [5, 5, 5], 0),                 # tất cả vào được
    ([4, 2, 5, 1], [1, 5, 2, 4], 1),           # 4->5, 2->2, 5->không, 1->1
]
for fr, bk, exp in tests:
    assert len(fr) == len(bk) and 1 <= len(fr) <= 100
    assert all(1 <= v <= 1000 for v in fr + bk)
    for im in impls:
        got = im.numOfUnplacedFruits(list(fr), list(bk))
        assert got == exp, (type(im).__name__, fr, bk, got, exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 12


Test 2: ngẫu nhiên có seed, đúng constraints (`1 <= n <= 100`, giá trị `1..1000`). Oracle độc lập mô phỏng bằng **dictionary** `chỉ_số -> sức chứa`, xoá khoá khi dùng (cách viết khác hẳn các cách trên).

In [15]:
import random
random.seed(3477)

def oracle(fruits, baskets):
    free = {j: c for j, c in enumerate(baskets)}
    cnt = 0
    for f in fruits:
        cands = [j for j, c in free.items() if c >= f]
        if cands:
            del free[min(cands)]
        else:
            cnt += 1
    return cnt

for t in range(1500):
    n = random.randint(1, 100) if t % 50 == 0 else random.randint(1, 10)
    hi_val = random.choice([5, 20, 1000])
    fr = [random.randint(1, hi_val) for _ in range(n)]
    bk = [random.randint(1, hi_val) for _ in range(n)]
    assert 1 <= n <= 100 and all(1 <= v <= 1000 for v in fr + bk)
    exp = oracle(fr, bk)
    for im in impls:
        assert im.numOfUnplacedFruits(list(fr), list(bk)) == exp, (type(im).__name__, fr, bk)
print("1500 test ngẫu nhiên OK")

1500 test ngẫu nhiên OK


Test 3: tính chất — không sửa input; kết quả luôn trong `[0, n]`; nếu mọi sức chứa `>=` mọi số lượng thì đáp án `0`; nếu mọi sức chứa `<` mọi số lượng thì đáp án `n`; **thêm** một giỏ khổng lồ (thay giỏ cuối bằng `1000`) không làm đáp án tăng.

In [16]:
random.seed(77)
for _ in range(500):
    n = random.randint(1, 30)
    fr = [random.randint(1, 500) for _ in range(n)]
    bk = [random.randint(1, 500) for _ in range(n)]
    fs, bs = list(fr), list(bk)
    ans = Solution().numOfUnplacedFruits(fr, bk)
    assert fr == fs and bk == bs
    assert 0 <= ans <= n
    assert Solution().numOfUnplacedFruits(fr, [1000] * n) == 0
    assert Solution().numOfUnplacedFruits([1000] * n, [999] * n) == n
    bk2 = list(bk)
    bk2[-1] = 1000
    assert Solution().numOfUnplacedFruits(fr, bk2) <= ans
print("tính chất (không sửa input, [0,n], biên, giỏ lớn không làm tệ hơn): OK")

tính chất (không sửa input, [0,n], biên, giỏ lớn không làm tệ hơn): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: không đánh dấu giỏ đã dùng (dùng lại một giỏ nhiều lần)

In [17]:
def wrong_reuse(fruits, baskets):
    unplaced = 0
    for f in fruits:
        if not any(c >= f for c in baskets):    # SAI: quên đánh dấu giỏ đã dùng
            unplaced += 1
    return unplaced

fr, bk = [2, 2], [3, 1]
print("sai :", wrong_reuse(fr, bk))
print("đúng:", Solution().numOfUnplacedFruits(fr, bk))
assert wrong_reuse(fr, bk) != Solution().numOfUnplacedFruits(fr, bk)

sai : 0
đúng: 1


Vì sao sai: giỏ `3` được dùng cho quả đầu rồi **lại** dùng cho quả thứ hai nên đáp án `0`, trong khi mỗi giỏ chỉ chứa một loại: loại thứ hai không có giỏ (giỏ `1` quá nhỏ) -> đáp án thật là `1`. **Sửa:** `used[j] = True` sau khi chọn.

### Lỗi 2: chọn giỏ nhỏ nhất vừa đủ (best-fit) thay vì giỏ trái nhất

In [18]:
def wrong_best_fit(fruits, baskets):
    free = list(baskets)
    unplaced = 0
    for f in fruits:
        cands = [c for c in free if c >= f]
        if cands:
            free.remove(min(cands))             # SAI: đề yêu cầu giỏ TRÁI NHẤT, không phải giỏ nhỏ nhất
        else:
            unplaced += 1
    return unplaced

fr, bk = [3, 5], [5, 4]
print("sai :", wrong_best_fit(fr, bk))
print("đúng:", Solution().numOfUnplacedFruits(fr, bk))
assert wrong_best_fit(fr, bk) != Solution().numOfUnplacedFruits(fr, bk)

sai : 0
đúng: 1


Vì sao sai: quả `3` theo đề vào giỏ `5` (giỏ trái nhất đủ lớn), nên quả `5` hết giỏ -> đáp án `1`. Best-fit đưa `3` vào giỏ `4` và để giỏ `5` cho quả `5` -> `0`. Quy tắc của đề là **vị trí**, không phải kích thước. **Sửa:** quét giỏ theo chỉ số tăng dần và dừng ở giỏ đầu tiên thoả mãn.

### Lỗi 3: sắp xếp cả hai mảng rồi ghép

In [19]:
def wrong_sorted_match(fruits, baskets):
    fs, bs = sorted(fruits), sorted(baskets)    # SAI: sắp xếp làm mất thứ tự trái -> phải
    return sum(1 for f, b in zip(fs, bs) if f > b)

fr, bk = [3, 5], [5, 4]
print("sai :", wrong_sorted_match(fr, bk))
print("đúng:", Solution().numOfUnplacedFruits(fr, bk))
assert wrong_sorted_match(fr, bk) != Solution().numOfUnplacedFruits(fr, bk)

sai : 0
đúng: 1


Vì sao sai: sắp xếp là chiến thuật ghép tối ưu cho bài toán "tối đa số cặp", nhưng đề này có **quy tắc cố định** (trái nhất đủ lớn). Sắp xếp xoá mất chỉ số nên ra `0` thay vì `1`. **Sửa:** giữ nguyên thứ tự hai mảng.

### Lỗi 4: chỉ so `fruits[i]` với `baskets[i]` cùng vị trí

In [20]:
def wrong_same_index(fruits, baskets):
    return sum(1 for f, b in zip(fruits, baskets) if f > b)   # SAI: quả i có thể vào giỏ j != i

fr, bk = [4, 2, 5], [3, 5, 4]
print("sai :", wrong_same_index(fr, bk))
print("đúng:", Solution().numOfUnplacedFruits(fr, bk))
assert wrong_same_index(fr, bk) != Solution().numOfUnplacedFruits(fr, bk)

sai : 2
đúng: 1


Vì sao sai: ở ví dụ 1, quả `4` (vị trí 0) không vào giỏ `3` (vị trí 0) nhưng vào được giỏ `5` (vị trí 1). So cùng chỉ số cho `2` loại không xếp được, đáp án thật là `1`. **Sửa:** với mỗi quả, tìm trong **tất cả** các giỏ chưa dùng.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề có từ khoá "**trái nhất**", "đầu tiên thoả mãn", "mỗi phần tử dùng một lần" -> mô phỏng + mảng `used` (hoặc xoá khỏi danh sách).
- "Sức chứa >= số lượng" là điều kiện **không nghiêm ngặt** (`>=`), đừng viết `>`.
- Thấy "tìm vị trí trái nhất có giá trị >= x" lặp lại với cập nhật -> **segment tree lưu max** (hoặc sqrt decomposition) khi `n` lớn.
- Với `n <= 100` đừng over-engineer: hai vòng lặp là đủ và dễ đúng.
- Phân biệt với bài "ghép tối đa" (greedy sắp xếp): ở đây thứ tự là luật, không được sắp xếp.
- Kiểm tra: `n = 1`, quả lớn hơn mọi giỏ, tất cả vừa khít.

## 15. Biến thể của bài

1. **Fruits Into Baskets III** (`3479`): `n` tới `10^5` -> cần segment tree / cấu trúc tìm trái nhất `O(log n)`.
2. **Best-fit**: chọn giỏ nhỏ nhất vẫn đủ (dùng `bisect` trên danh sách đã sắp xếp hoặc `SortedList`).
3. **Một giỏ chứa nhiều loại** nếu tổng không vượt sức chứa (dạng bin packing — first-fit).
4. Trả về **danh sách chỉ số giỏ** của từng quả (`-1` nếu không xếp).
5. Đếm tổng số quả **được** xếp (cộng `fruits[i]` các loại đã xếp) thay vì số loại bị bỏ.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai vòng lặp + `used` | quét tuyến tính, đánh dấu giỏ | `O(n^2)` | `O(n)` | **chuẩn, nên nộp** với `n <= 100` |
| 2. Bản sao + `pop` | xoá giỏ đã dùng khỏi danh sách | `O(n^2)` | `O(n)` | gọn, dễ đọc |
| 3. Đánh dấu `0` + `next` | giỏ dùng rồi có sức chứa `0` | `O(n^2)` | `O(n)` | không cần mảng phụ `used` |
| 4. Segment tree `max` | xuống cây tìm trái nhất đủ lớn | `O(n log n)` | `O(n)` | `n` lớn (bài III) |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `numOfUnplacedFruits([5, 3, 4], [4, 6, 3])` bằng bao nhiêu?

**Bài 2 (tính tay):** `fruits = [2, 8, 3]`, `baskets = [9, 2, 4]`: quả nào vào giỏ nào, bao nhiêu loại không xếp?

**Bài 3 (điền TODO):** hoàn thành `first_fit(baskets, used, f)` trả về chỉ số giỏ trống trái nhất có sức chứa `>= f`, hoặc `-1`.

**Bài 4 (tự viết lại):** viết `assign_baskets(fruits, baskets)` trả về danh sách chỉ số giỏ của từng quả (`-1` nếu không xếp).

**Bài 5 (mini challenge):** viết `placed_quantity(fruits, baskets)` trả về **tổng số lượng quả** đã được xếp vào giỏ.

In [21]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().numOfUnplacedFruits([5, 3, 4], [4, 6, 3]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def first_fit(baskets, used, f):
    # TODO: duyệt j từ 0; trả về j đầu tiên có (not used[j]) và baskets[j] >= f; không có thì -1
    return -1

1


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** với mỗi quả, nhìn các giỏ từ trái sang phải, bỏ qua giỏ đã dùng, nhận giỏ đầu tiên có sức chứa không nhỏ hơn số lượng.

**Mức 2 (vừa):** Bài 3: `for j in range(len(baskets))` rồi `if not used[j] and baskets[j] >= f: return j`; cuối hàm `return -1`. Bài 4 dùng `first_fit`, nhớ đặt `used[j] = True`. Bài 5 cộng `f` mỗi khi `first_fit` khác `-1`.

**Mức 3 (gần đáp án):** Bài 1: `5` -> giỏ `6`; `3` -> giỏ `4`; `4` -> giỏ `3` không đủ, hết giỏ -> đáp án `1`. Bài 2: `2` -> giỏ `9`; `8` -> không (còn `2`, `4`); `3` -> giỏ `4` -> đáp án `1`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1: 5->giỏ 6 ; 3->giỏ 4 ; 4->(giỏ 3 < 4) không có -> 1
assert Solution().numOfUnplacedFruits([5, 3, 4], [4, 6, 3]) == 1

# Bài 2: 2->giỏ 9 ; 8->không ; 3->giỏ 4 -> 1
assert Solution().numOfUnplacedFruits([2, 8, 3], [9, 2, 4]) == 1

# Bài 3
def first_fit(baskets, used, f):
    for j in range(len(baskets)):
        if not used[j] and baskets[j] >= f:
            return j
    return -1
assert first_fit([3, 5, 4], [False, False, False], 4) == 1
assert first_fit([3, 5, 4], [False, True, False], 4) == 2
assert first_fit([3, 5, 4], [False, True, True], 4) == -1

# Bài 4
def assign_baskets(fruits, baskets):
    used = [False] * len(baskets)
    out = []
    for f in fruits:
        j = first_fit(baskets, used, f)
        if j != -1:
            used[j] = True
        out.append(j)
    return out
assert assign_baskets([4, 2, 5], [3, 5, 4]) == [1, 0, -1]
assert assign_baskets([3, 6, 1], [6, 4, 7]) == [0, 2, 1]
assert assign_baskets([2, 8, 3], [9, 2, 4]) == [0, -1, 2]

# Bài 5
def placed_quantity(fruits, baskets):
    return sum(f for f, j in zip(fruits, assign_baskets(fruits, baskets)) if j != -1)
assert placed_quantity([4, 2, 5], [3, 5, 4]) == 6
assert placed_quantity([3, 6, 1], [6, 4, 7]) == 10
for fr, bk in (([4, 2, 5], [3, 5, 4]), ([5, 5], [1, 1])):
    assert assign_baskets(fr, bk).count(-1) == Solution().numOfUnplacedFruits(fr, bk)
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "mỗi phần tử lấy phần tử trái nhất thoả điều kiện, dùng một lần":**

```python
used = [False] * len(baskets)
unplaced = 0
for f in fruits:
    for j in range(len(baskets)):
        if not used[j] and baskets[j] >= f:
            used[j] = True
            break
    else:                       # không break: không tìm được giỏ
        unplaced += 1
return unplaced
```

**Checklist trước khi submit:**

- [ ] Duyệt quả từ trái sang phải, giỏ từ trái sang phải, chọn giỏ **trái nhất** đủ lớn (không best-fit)?
- [ ] Điều kiện `baskets[j] >= f` (không phải `>`)?
- [ ] Đã đánh dấu giỏ đã dùng (và `break` sau khi chọn)?
- [ ] Không so `fruits[i]` với `baskets[i]` cùng chỉ số một cách máy móc?
- [ ] Không sắp xếp lại hai mảng?
- [ ] Đã thử `n = 1`, quả lớn hơn mọi giỏ, mọi quả vào được?
- [ ] Chữ ký đúng `numOfUnplacedFruits(self, fruits: List[int], baskets: List[int]) -> int`?